In [ ]:
import pandas as pd
import requests
import time
import os
import re
from datetime import datetime
from sqlalchemy import create_engine, text
import urllib3

# [1] SSL 인증서 경로 에러 방지 및 경고 무시
for env_var in ['REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE']:
    if env_var in os.environ:
        del os.environ[env_var]
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- [설정 정보] ---
KAKAO_API_KEY = "secret"
DB_USER, DB_PASS = "root", "1234"
DB_HOST, DB_PORT, DB_NAME = "localhost", "3306", "coffee_store"
CSV_FILE = "dunkindonuts.csv"

def clean_address(addr):
    """주소에서 지도 API가 인식하기 힘든 상세 정보 제거"""
    if not addr or pd.isna(addr): return ""
    # 괄호 및 내용 제거 (예: (전농동))
    addr = re.sub(r'\(.*?\)', '', addr)
    # 상세 층수 및 호수 제거 (예: 1층, 지하2층, 101호)
    addr = re.sub(r'\d+층|\d+호|지하\d+층|지상\d+층', '', addr)
    return " ".join(addr.split())

def get_kakao_coords(address):
    """카카오 API 호출"""
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params, timeout=10, verify=False)
        if res.status_code == 200:
            data = res.json()
            if data['documents']:
                return float(data['documents'][0]['y']), float(data['documents'][0]['x'])
        return None, None
    except:
        return None, None

def run_failed_data_recovery():
    if not os.path.exists(CSV_FILE):
        print(f"❌ '{CSV_FILE}' 파일이 없습니다.")
        return

    # 1. 데이터 로드
    df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    # '위도' 컬럼이 없으면 생성, 있으면 빈 값 확인
    if '위도' not in df.columns:
        df['위도'] = None
        df['경도'] = None

    # 2. 실패한 데이터만 필터링 (NaN 혹은 'None' 문자열 등)
    failed_mask = df['위도'].isna() | (df['위도'] == '') | (df['위도'].astype(str) == 'None')
    failed_indices = df[failed_mask].index

    if len(failed_indices) == 0:
        print("✅ 모든 데이터에 이미 좌표가 존재합니다.")
        return

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🛠️ 실패 데이터 {len(failed_indices)}건에 대해 카카오 API 보정을 시작합니다.")
    print("-" * 70)

    # 3. 보정 작업 수행
    for i in failed_indices:
        store_nm = df.at[i, '매장명']
        original_addr = df.at[i, '주소']
        
        # 주소 정제 후 API 호출
        refined_addr = clean_address(original_addr)
        lat, lon = get_kakao_coords(refined_addr)
        
        # 1차 실패 시 주소를 더 짧게 잘라서 재시도 (건물명 제외)
        if lat is None:
            short_addr = " ".join(refined_addr.split()[:4])
            lat, lon = get_kakao_coords(short_addr)

        if lat:
            df.at[i, '위도'] = lat
            df.at[i, '경도'] = lon
            status = "✅ 보정성공"
        else:
            status = "❌ 최종실패"

        now = datetime.now().strftime('%H:%M:%S')
        print(f"[{now}] ({i+1}/{len(df)}) {store_nm[:10]:<10} | {status} | 주소: {refined_addr[:25]}...")
        
        time.sleep(0.05)

    # 4. 결과 저장
    df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
    print("-" * 70)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ 보정 작업 완료 및 CSV 저장.")

if __name__ == "__main__":
    run_failed_data_recovery()

[14:30:40] 🛠️ 실패 데이터 625건에 대해 카카오 API 보정을 시작합니다.
----------------------------------------------------------------------
[14:30:41] (1/625) 북창         | ✅ 보정성공 | 주소: 서울 중구 남대문로1길 60...
[14:30:41] (2/625) 염창         | ✅ 보정성공 | 주소: 서울 강서구 양천로 723...
[14:30:41] (3/625) 구로태평양물산    | ✅ 보정성공 | 주소: 서울 구로구 디지털로31길 12 102동...
[14:30:41] (4/625) 선릉역        | ✅ 보정성공 | 주소: 서울 강남구 테헤란로 422...
[14:30:42] (5/625) 원더스 강남     | ✅ 보정성공 | 주소: 서울 서초구 강남대로 373...
[14:30:42] (6/625) 왕십리민자역사    | ✅ 보정성공 | 주소: 서울 성동구 왕십리광장로 17 , 왕십리광장...
[14:30:43] (7/625) 양재사옥       | ✅ 보정성공 | 주소: 서울 서초구 남부순환로 2620...
[14:30:43] (8/625) 고덕역        | ✅ 보정성공 | 주소: 서울 강동구 동남로73길 17...
[14:30:43] (9/625) 테크노마트      | ✅ 보정성공 | 주소: 서울 광진구 광나루로56길 85...
[14:30:43] (10/625) 암사역        | ✅ 보정성공 | 주소: 서울 강동구 올림픽로 775...
[14:30:44] (11/625) 등촌         | ✅ 보정성공 | 주소: 서울 강서구 공항대로 335...
[14:30:44] (12/625) 청담         | ✅ 보정성공 | 주소: 서울 강남구 삼성로 651...
[14:30:44] (13/625) 응암역        | ✅ 보정성공 | 주소: 서울 은평구 은평로 85 A-101...
[14:30:45] (14/625) 영

In [31]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time
import os

def fill_dunkindonuts_coords(file_path):
    # 1. 데이터 로드 및 위도/경도 컬럼 준비
    if not os.path.exists(file_path):
        print(f"❌ {file_path} 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")
        return

    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # '위도'와 '경도' 컬럼이 없으면 새로 생성 (빈 값으로 초기화)
    if '위도' not in df.columns:
        df['위도'] = None
    if '경도' not in df.columns:
        df['경도'] = None
    
    # 위도 또는 경도가 없는 행만 추출 (NaN 확인)
    missing_mask = df['위도'].isna() | df['경도'].isna()
    missing_df = df[missing_mask].copy()
    
    if missing_df.empty:
        print("✅ 모든 파스쿠찌 매장의 좌표가 이미 존재합니다.")
        return df

    print(f"📡 총 {len(missing_df)}개의 빈 좌표를 발견했습니다. 수집을 시작합니다.")

    # 2. Geopy 및 RateLimiter 설정
    # Nominatim은 무료이므로 지연 시간(min_delay_seconds)을 넉넉히(1.5초 이상) 주는 것이 중요합니다.
    geolocator = Nominatim(user_agent="dunkindonuts_korea_geocoder_v1")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

    # 3. 루프 실행 및 좌표 채우기
    start_time = time.time()
    success_count = 0
    fail_count = 0

    for idx, row in missing_df.iterrows():
        # 디저트39 CSV의 주소 컬럼 사용
        address = row['주소']
        try:
            # 1단계: 전체 주소로 검색 시도
            location = geocode(address)
            
            # 2단계: 실패 시 주소를 앞 3단어(시/군/구)로 잘라서 재시도 (fallback)
            if not location:
                short_addr = " ".join(address.split()[:3])
                location = geocode(short_addr)
            
            if location:
                df.at[idx, '위도'] = location.latitude
                df.at[idx, '경도'] = location.longitude
                success_count += 1
                print(f"✔️ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 위치 확보")
            else:
                fail_count += 1
                print(f"❌ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 좌표 찾기 실패")
                
        except Exception as e:
            fail_count += 1
            print(f"❗ [{success_count + fail_count}/{len(missing_df)}] {row['매장명']} 오류 발생: {e}")

    # 4. 결과 저장
    # 작업 중간에 멈춰도 데이터를 잃지 않도록 덮어쓰기 방식으로 저장합니다.
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print(f"✨ 매머드커피 좌표 수집 완료! (소요 시간: {elapsed:.2f}초)")
    print(f"✅ 성공: {success_count}건 | ❌ 실패: {fail_count}건")
    print(f"📂 결과가 {file_path}에 최종 업데이트되었습니다.")
    print("="*60)

    return df

# 실행 부분
if __name__ == "__main__":
    # VS Code 프로젝트 폴더 안에 ediya.csv가 있어야 합니다.
    fill_dunkindonuts_coords('dunkindonuts.csv')

📡 총 14개의 빈 좌표를 발견했습니다. 수집을 시작합니다.
✔️ [1/14] 목동이마트 위치 확보
✔️ [2/14] 영등포역사2호 위치 확보
✔️ [3/14] 트레이더스 연산 위치 확보
❌ [4/14] 세종연구단지 좌표 찾기 실패
❌ [5/14] 세종고운 좌표 찾기 실패
❌ [6/14] 세종보람 좌표 찾기 실패
❌ [7/14] 세종나성 좌표 찾기 실패
❌ [8/14] 세종정부청사 좌표 찾기 실패
❌ [9/14] 세종아름 좌표 찾기 실패
❌ [10/14] 시흥하늘휴게소 좌표 찾기 실패
❌ [11/14] 시흥휴게소 좌표 찾기 실패
✔️ [12/14] 서하남휴게소 위치 확보
❌ [13/14] 의정부고산 좌표 찾기 실패
✔️ [14/14] 삼척중앙 위치 확보

✨ 매머드커피 좌표 수집 완료! (소요 시간: 38.36초)
✅ 성공: 5건 | ❌ 실패: 9건
📂 결과가 dunkindonuts.csv에 최종 업데이트되었습니다.


In [32]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. CSV 파일 로드
file_path = 'dunkindonuts.csv'
df = pd.read_csv(file_path)

# 2. MySQL 연결 설정 (사용자 정보에 맞게 수정 필요)
user = 'root'      # MySQL 사용자명 (예: root)
password = '1234'  # MySQL 비밀번호
host = 'localhost'          # 호스트 (예: 127.0.0.1)
port = '3306'               # 포트 번호
db_name = 'coffee_store'    # 대상 데이터베이스

# 3. 데이터베이스 생성 (이미 존재하면 무시)
# 먼저 MySQL 서버 자체에 연결하여 DB가 없으면 생성합니다.
temp_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")
with temp_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db_name}"))
    print(f"데이터베이스 '{db_name}'가 준비되었습니다.")

# 4. SQLAlchemy 엔진 생성 (coffee_store DB 연결)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 5. 데이터프레임을 MySQL 테이블로 저장
# - name: 생성할 테이블 이름 ('the_venti')
# - if_exists: 'fail' (이미 있으면 오류), 'replace' (기존 삭제 후 생성), 'append' (기존 테이블에 추가)
# - index: False (데이터프레임의 인덱스는 컬럼으로 넣지 않음)
try:
    df.to_sql(name='dunkindonuts', con=engine, if_exists='replace', index=False)
    print(f"테이블 'dunkindonuts'에 총 {len(df)}건의 데이터가 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"데이터 저장 중 오류 발생: {e}")

# 연결 종료
engine.dispose()

데이터베이스 'coffee_store'가 준비되었습니다.
테이블 'dunkindonuts'에 총 625건의 데이터가 성공적으로 저장되었습니다.
